In [310]:
#pip install -U kaleido
#pip install -U plotly
#pip install xlrd
#pip install pycountry


In [311]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import numpy as np


### Cargo la el dataset principal: ACLED   
Cada linea es un evento de conflicto.  

In [312]:
df = pd.read_csv('ACLED Data.csv', sep=';')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2708208 entries, 0 to 2708207
Data columns (total 32 columns):
 #   Column              Dtype  
---  ------              -----  
 0   event_id_cnty       object 
 1   event_date          object 
 2   year                int64  
 3   time_precision      int64  
 4   disorder_type       object 
 5   event_type          object 
 6   sub_event_type      object 
 7   actor1              object 
 8   assoc_actor_1       object 
 9   inter1              object 
 10  actor2              object 
 11  assoc_actor_2       object 
 12  inter2              object 
 13  interaction         object 
 14  civilian_targeting  object 
 15  iso                 int64  
 16  region              object 
 17  country             object 
 18  admin1              object 
 19  admin2              object 
 20  admin3              object 
 21  location            object 
 22  latitude            float64
 23  longitude           float64
 24  geo_precision       int6

Por un mejor almacenamiento, hago un astype int32 por todas las columnas años

In [313]:
df['year'] = df['year'].astype(int)

In [314]:
display(df.columns)

print('Event type: ',df['event_type'].unique())

print('Sub Event Type: ',df['sub_event_type'].unique())

Index(['event_id_cnty', 'event_date', 'year', 'time_precision',
       'disorder_type', 'event_type', 'sub_event_type', 'actor1',
       'assoc_actor_1', 'inter1', 'actor2', 'assoc_actor_2', 'inter2',
       'interaction', 'civilian_targeting', 'iso', 'region', 'country',
       'admin1', 'admin2', 'admin3', 'location', 'latitude', 'longitude',
       'geo_precision', 'source', 'source_scale', 'notes', 'fatalities',
       'tags', 'timestamp', 'population_1km'],
      dtype='object')

Event type:  ['Violence against civilians' 'Battles' 'Strategic developments'
 'Protests' 'Explosions/Remote violence' 'Riots']
Sub Event Type:  ['Attack' 'Armed clash' 'Arrests' 'Protest with intervention'
 'Abduction/forced disappearance' 'Suicide bomb'
 'Remote explosive/landmine/IED' 'Air/drone strike'
 'Disrupted weapons use' 'Peaceful protest' 'Sexual violence'
 'Mob violence' 'Shelling/artillery/missile attack'
 'Change to group/activity' 'Violent demonstration'
 'Looting/property destruction' 'Government regains territory'
 'Excessive force against protesters' 'Grenade'
 'Non-state actor overtakes territory' 'Non-violent transfer of territory'
 'Agreement' 'Headquarters or base established' 'Other' 'Chemical weapon']


Cuantos paises unico tiene el dataset? 

In [315]:
len(df['iso'].unique())

238

### Cargo datos de PIB por años desde World Bank Data https://databank.worldbank.org/source/world-development-indicators?savedlg=1&l=en,en#
## GDP per capita, PPP (constant 2021 international $)

In [316]:
wb_ind = pd.read_excel(
    'GDP.xlsx',
    na_values='..',
    skipfooter=5,
    engine='openpyxl'   
)

wb_ind.columns

wb_ind.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Series Name    266 non-null    object 
 1   Series Code    266 non-null    object 
 2   Country Name   266 non-null    object 
 3   Country Code   266 non-null    object 
 4   2015 [YR2015]  247 non-null    float64
 5   2016 [YR2016]  247 non-null    float64
 6   2017 [YR2017]  247 non-null    float64
 7   2018 [YR2018]  247 non-null    float64
 8   2019 [YR2019]  247 non-null    float64
 9   2020 [YR2020]  247 non-null    float64
 10  2021 [YR2021]  247 non-null    float64
 11  2022 [YR2022]  247 non-null    float64
 12  2023 [YR2023]  245 non-null    float64
 13  2024 [YR2024]  236 non-null    float64
dtypes: float64(10), object(4)
memory usage: 29.2+ KB


Cada linea es un pais en el dataset del Banco Mundial, asi que son 266 paises unicos.  
-> El codigo de pais es una referencia iso3

Por cada columna que es digit extraigo los primero 4

In [317]:
wb_ind.columns = [col[:4] if col[:4].isdigit() else col for col in wb_ind.columns]

In [318]:
wb_ind.head()

,Series Name,Series Code,Country Name,Country Code,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,Afghanistan,AFG,2967.692067,2958.785399,2952.998916,2902.392113,2927.245144,2769.685745,2144.166570,1981.710168,1983.812620,NaN
1,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,Albania,ALB,13203.796947,13933.283751,14614.531162,15386.701533,15948.278917,15659.588696,17329.550634,18448.517677,19487.710768,20591.486725
2,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,Algeria,DZA,15239.517146,15511.685877,15427.664189,15343.426122,15199.198997,14194.155748,14496.865470,14782.200297,15159.324237,15501.919700
3,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,American Samoa,ASM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,Andorra,AND,62105.154239,64402.809237,63239.657771,63048.598557,63215.899792,55488.490299,59332.202910,63913.383508,64631.296391,65928.307197


In [319]:
wb_ind = wb_ind.rename(columns={'Country Code': 'iso3'})

In [320]:
wb_ind

,Series Name,Series Code,Country Name,iso3,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,Afghanistan,AFG,2967.692067,2958.785399,2952.998916,2902.392113,2927.245144,2769.685745,2144.166570,1981.710168,1983.812620,NaN
1,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,Albania,ALB,13203.796947,13933.283751,14614.531162,15386.701533,15948.278917,15659.588696,17329.550634,18448.517677,19487.710768,20591.486725
2,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,Algeria,DZA,15239.517146,15511.685877,15427.664189,15343.426122,15199.198997,14194.155748,14496.865470,14782.200297,15159.324237,15501.919700
3,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,American Samoa,ASM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,Andorra,AND,62105.154239,64402.809237,63239.657771,63048.598557,63215.899792,55488.490299,59332.202910,63913.383508,64631.296391,65928.307197
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,Sub-Saharan Africa,SSF,5016.227530,4948.610021,4944.827166,4954.562288,4956.282046,4674.729995,4720.357614,4796.066415,4810.215768,4872.635614
262,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,Sub-Saharan Africa (excluding high income),SSA,5358.655619,5268.831501,5249.094983,5249.943568,5237.578593,4915.930793,4958.956150,5035.769108,5040.154179,5094.613125
263,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,Sub-Saharan Africa (IDA & IBRD countries),TSS,5016.227530,4948.610021,4944.827166,4954.562288,4956.282046,4674.729995,4720.357614,4796.066415,4810.215768,4872.635614
264,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,Upper middle income,UMC,15969.576877,16537.893269,17239.461385,17902.247354,18488.219862,18237.347144,19501.635678,20089.079982,20944.239955,21731.496873


In [321]:
year_cols = ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

pib_largo = wb_ind.melt(
    id_vars=['iso3'],
    value_vars=year_cols,
    var_name='year',
    value_name='gdp_pc'
)

pib_largo['year'] = pib_largo['year'].astype(int)
pib_largo['gdp_pc'] = pib_largo['gdp_pc'].round(2)

In [322]:
pib_largo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2660 entries, 0 to 2659
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   iso3    2660 non-null   object 
 1   year    2660 non-null   int32  
 2   gdp_pc  2457 non-null   float64
dtypes: float64(1), int32(1), object(1)
memory usage: 52.1+ KB


## GINI

In [323]:
gini = pd.read_excel(
    'GINI.xlsx',
    skiprows=0,
    skipfooter=5,
    na_values='..',
    engine='openpyxl'
)


In [324]:
gini.head()

,Series Name,Series Code,Country Name,Country Code,2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024]
0,Gini index,SI.POV.GINI,Afghanistan,AFG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Gini index,SI.POV.GINI,Albania,ALB,32.8,33.7,33.1,30.1,30.1,29.4,NaN,NaN,NaN,NaN
2,Gini index,SI.POV.GINI,Algeria,DZA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Gini index,SI.POV.GINI,American Samoa,ASM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Gini index,SI.POV.GINI,Andorra,AND,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Repito la misma logica que he aplicado anteriormente con las columnas de GDP

In [325]:
gini.columns = [col[:4] if col[:4].isdigit() else col for col in gini.columns]

In [326]:
gini.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Series Name   266 non-null    object 
 1   Series Code   266 non-null    object 
 2   Country Name  266 non-null    object 
 3   Country Code  266 non-null    object 
 4   2015          87 non-null     float64
 5   2016          83 non-null     float64
 6   2017          78 non-null     float64
 7   2018          93 non-null     float64
 8   2019          78 non-null     float64
 9   2020          70 non-null     float64
 10  2021          81 non-null     float64
 11  2022          73 non-null     float64
 12  2023          57 non-null     float64
 13  2024          25 non-null     float64
dtypes: float64(10), object(4)
memory usage: 29.2+ KB


Cambio el tipo de columnas para no tener problemas a la hora del merge

In [327]:
year_cols = gini.columns[gini.columns.str.match(r'^\d{4}')]
gini[year_cols] = gini[year_cols].apply(pd.to_numeric, errors='coerce')

In [328]:
gini = gini.rename(columns={'Country Code': 'iso3', 'ALL': 'gini'})

In [329]:
gini_largo = gini.melt(
    id_vars=['iso3'],
    value_vars=year_cols,
    var_name='year',
    value_name='gini'
)

gini_largo['year'] = gini_largo['year'].astype(int)

In [330]:
gini_largo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2660 entries, 0 to 2659
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   iso3    2660 non-null   object 
 1   year    2660 non-null   int32  
 2   gini    725 non-null    float64
dtypes: float64(1), int32(1), object(1)
memory usage: 52.1+ KB


## POB

In [331]:
pob = pd.read_csv('POB.csv', skipfooter=5, na_values='..', engine='python')

In [332]:
pob = pob.rename(columns={
    'Population, total [SP.POP.TOTL]': 'population', 
    'Country Code': 'iso3',
    'Time':'year'})

In [333]:
pob['year'] = pob['Time Code'].str.extract(r'(\d{4})').astype(int)

In [334]:
pob['population'] = pob['population'].astype(float)

In [335]:
pob_clean = pob[['iso3', 'year', 'population']] 

In [336]:
pob_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2660 entries, 0 to 2659
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   iso3        2660 non-null   object 
 1   year        2660 non-null   int32  
 2   population  2650 non-null   float64
dtypes: float64(1), int32(1), object(1)
memory usage: 52.1+ KB


In [337]:
pob_clean

,iso3,year,population
0,AFG,2015,3.383176e+07
1,ALB,2015,2.731293e+06
2,DZA,2015,4.001953e+07
3,ASM,2015,5.287800e+04
4,AND,2015,7.217400e+04
...,...,...,...
2655,SSF,2024,1.291045e+09
2656,SSA,2024,1.158864e+09
2657,TSS,2024,1.291045e+09
2658,UMC,2024,2.817913e+09


In [338]:
cont_df = df['iso'].nunique()
print('Paises en ACLED : ',cont_df)
cont_wb = wb_ind['iso3'].nunique()
print('Paises en DF GDP : ',cont_wb)

df[['iso']].head()


Paises en ACLED :  238
Paises en DF GDP :  266


,iso
0,894
1,50
2,270
3,704
4,108


## Indicatores WGI:  
### 1) PV (Political Stability)  
¿Es estable el gobierno? / ¿Hay riesgo de violencia política?  
### 2) RL (Rule of Law)
¿Funciona la justicia y se respetan las leyes?  


In [339]:
pv = pd.read_excel('wgidataset_with_sourcedata-2025.xlsx', sheet_name='pv')
rl = pd.read_excel('wgidataset_with_sourcedata-2025.xlsx', sheet_name='rl')

In [340]:
pv.columns = pv.columns.str.strip()

pv.columns

Index(['ID variable (economy code/ gov. dimension/ year)', 'Economy (name)',
       'Economy (code)', 'Region', 'Income classification', 'Year',
       'Governance dimension', 'Number of sources',
       'Governance estimate (approx. -2.5 to +2.5)',
       'Standard error (estimate)',
       'Lower threshold (90% conf. int. estimate)',
       'Upper threshold (90% conf. int. estimate)', 'Governance score (0-100)',
       'Standard error (gov. score)', 'Lower threshold (90% conf. int. score)',
       'Upper threshold (90% conf. int. score)', 'ADB mean', 'AFR mean',
       'ARB mean', 'ASB mean', 'ASD mean', 'BPS mean', 'BTI mean', 'CCR mean',
       'EBR mean', 'EIU mean', 'EQI mean', 'EUB mean', 'FRH mean', 'GCB mean',
       'GCS mean', 'GII mean', 'GWP mean', 'HER mean', 'HRM mean', 'HUM mean',
       'IFD mean', 'IJT mean', 'IRP mean', 'LBO mean', 'MSI mean', 'OBI mean',
       'PIA mean', 'PRS mean', 'RSF mean', 'VAB mean', 'VDM mean', 'WBS mean',
       'WCY mean', 'WJP mean', 'WM

In [341]:
rl.columns = rl.columns.str.strip()

rl.columns

Index(['ID variable (economy code/ gov. dimension/ year)', 'Economy (name)',
       'Economy (code)', 'Region', 'Income classification', 'Year',
       'Governance dimension', 'Number of sources',
       'Governance estimate (approx. -2.5 to +2.5)',
       'Standard error (estimate)',
       'Lower threshold (90% conf. int. estimate)',
       'Upper threshold (90% conf. int. estimate)', 'Governance score (0-100)',
       'Standard error (gov. score)', 'Lower threshold (90% conf. int. score)',
       'Upper threshold (90% conf. int. score)', 'ADB mean', 'AFR mean',
       'ARB mean', 'ASB mean', 'ASD mean', 'BPS mean', 'BTI mean', 'CCR mean',
       'EBR mean', 'EIU mean', 'EQI mean', 'EUB mean', 'FRH mean', 'GCB mean',
       'GCS mean', 'GII mean', 'GWP mean', 'HER mean', 'HRM mean', 'HUM mean',
       'IFD mean', 'IJT mean', 'IRP mean', 'LBO mean', 'MSI mean', 'OBI mean',
       'PIA mean', 'PRS mean', 'RSF mean', 'VAB mean', 'VDM mean', 'WBS mean',
       'WCY mean', 'WJP mean', 'WM

In [342]:
pv_clean = pv[['Economy (code)', 'Year', 'Governance estimate (approx. -2.5 to +2.5)', 'Governance score (0-100)']].rename(columns={
    'Economy (code)': 'iso3',
    'Year': 'year',
    'Governance estimate (approx. -2.5 to +2.5)': 'political_stability',
    'Governance score (0-100)': 'political_stability_score'
})

rl_clean = rl[['Economy (code)', 'Year', 'Governance estimate (approx. -2.5 to +2.5)', 'Governance score (0-100)']].rename(columns={
    'Economy (code)': 'iso3',
    'Year': 'year',
    'Governance estimate (approx. -2.5 to +2.5)': 'rule_of_law',
    'Governance score (0-100)': 'rule_of_law_score'
})

In [343]:
rl_clean

,iso3,year,rule_of_law,rule_of_law_score
0,ADO,1996,0.991435,74.028006
1,AFG,1996,-1.917095,23.970162
2,AGO,1996,-1.570214,29.940223
3,ALB,1996,-0.394601,50.173356
4,ARE,1996,0.626891,67.753947
...,...,...,...,...
5471,XKX,2024,0.052136,57.862011
5472,YEM,2024,-1.901590,24.237016
5473,ZAF,2024,0.017703,57.269391
5474,ZMB,2024,-0.561177,47.306458


## Convierto el country code de pais (ACLED) en el estandard de WB iso3

In [344]:
import pycountry

def iso_num_to_iso3(x):
    try:
        return pycountry.countries.get(numeric=str(int(x)).zfill(3)).alpha_3
    except:
        return None

df['iso3'] = df['iso'].apply(iso_num_to_iso3)


### Kosovo y Akrotiri and Dhekelia son exepciones que he tenido que corregir

In [345]:
df['iso3'] = df['iso'].apply(iso_num_to_iso3)
df.loc[df['iso'].isin([0, 2]), 'iso3'] = 'XKX'

In [346]:
df['iso3'].isna().sum()

0

In [347]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2708208 entries, 0 to 2708207
Data columns (total 33 columns):
 #   Column              Dtype  
---  ------              -----  
 0   event_id_cnty       object 
 1   event_date          object 
 2   year                int32  
 3   time_precision      int64  
 4   disorder_type       object 
 5   event_type          object 
 6   sub_event_type      object 
 7   actor1              object 
 8   assoc_actor_1       object 
 9   inter1              object 
 10  actor2              object 
 11  assoc_actor_2       object 
 12  inter2              object 
 13  interaction         object 
 14  civilian_targeting  object 
 15  iso                 int64  
 16  region              object 
 17  country             object 
 18  admin1              object 
 19  admin2              object 
 20  admin3              object 
 21  location            object 
 22  latitude            float64
 23  longitude           float64
 24  geo_precision       int6

### finalmente hago el merge en un unico dataset

In [348]:
# Verifica righe prima del merge
print("Righe iniziali:", len(df))

Righe iniziali: 2708208


In [349]:
df = (df
    .merge(pib_largo, on=['iso3', 'year'], how='left')
    .merge(gini_largo, on=['iso3', 'year'], how='left')
    .merge(pob_clean, on=['iso3', 'year'], how='left')
    .merge(pv_clean, on=['iso3', 'year'], how='left')
    .merge(rl_clean, on=['iso3', 'year'], how='left')
)

In [350]:
print("Righe dopo merge:", len(df))
print("Columnas:", df.columns.tolist())

Righe dopo merge: 2708208
Columnas: ['event_id_cnty', 'event_date', 'year', 'time_precision', 'disorder_type', 'event_type', 'sub_event_type', 'actor1', 'assoc_actor_1', 'inter1', 'actor2', 'assoc_actor_2', 'inter2', 'interaction', 'civilian_targeting', 'iso', 'region', 'country', 'admin1', 'admin2', 'admin3', 'location', 'latitude', 'longitude', 'geo_precision', 'source', 'source_scale', 'notes', 'fatalities', 'tags', 'timestamp', 'population_1km', 'iso3', 'gdp_pc', 'gini', 'population', 'political_stability', 'political_stability_score', 'rule_of_law', 'rule_of_law_score']


## Añado las dos ultimas columnas calculada y extraigo el filde de data cleaning: 

In [351]:
df['has_fatalities'] = (df['fatalities'] > 0).astype(int)

In [352]:
df['fat_per_cap'] = df['fatalities'] / df['population']

---


In [353]:
df.to_csv('DATA_CLEANING_ACLED.csv', index=False)